# 3D Exploration - Monocular Depth Estimation

In this notebook, we'll use a pre-trained **MiDaS** model from PyTorch Hub to estimate depth from a single 2D image.

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Ensure matplotlib shows plots inline
%matplotlib inline

## 1. Load the Model
We load the MiDaS model and the appropriate transformations.

In [ ]:
model_type = "DPT_Large"

midas = torch.hub.load("intel-isl/MiDaS", model_type)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
midas.to(device)
midas.eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

## 2. Prepare the Image
Make sure `sample_image.jpeg` exists in the parent directory.

In [ ]:
image_path = "../sample_image.jpeg"
if not os.path.exists(image_path):
    print(f"Please upload an image to {image_path}")
else:
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    input_batch = transform(img).to(device)
    print("Image prepared.")

## 3. Run Inference

In [ ]:
if os.path.exists(image_path):
    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    
    output = prediction.cpu().numpy()
    print("Inference complete.")

## 4. Visualize the Results

In [ ]:
if os.path.exists(image_path):
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(img)
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.title("Depth Map")
    plt.imshow(output, cmap='inferno')
    plt.axis("off")
    
    plt.show()